# Week 12 — Capstone: End-to-End Domain-Specific Language Model

A single sprint that ties together every preceding week. Each student picks a domain (legal, medical, scientific, code, or a low-resource language) and delivers an end-to-end pipeline: data collection, tokenizer training, pretraining or continued pretraining, fine-tuning, evaluation, and a short report.

## Learning Objectives

- Design and execute a focused, end-to-end NLP project on a domain of your choice.
- Make principled choices about tokenizer, architecture, training data, and evaluation harness.
- Write a research-style report including ablations and an honest failure analysis.
- Defend the work in a 15-minute presentation.

## Structure

A 7-day sprint with three checkpoints:

| Day | Checkpoint | Deliverable |
|-----|-----------|-------------|
| 2   | Proposal + data pipeline | 1-page proposal, working data loader, EDA notebook |
| 4   | Trained baseline + eval harness | Reproducible baseline numbers with confidence intervals |
| 7   | Final report + presentation | 6–10 page report, presentation slides, clean repository |

In [ ]:
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
np.random.seed(0)

## 1. Suggested project tracks

You may propose your own track at Checkpoint 1, subject to approval. The three pre-vetted tracks below all admit a strong submission in the time budget.

### Track A — Domain pretraining
Continue pretraining a small open LM (e.g. a 100–300M parameter model) on a focused domain corpus — PubMed abstracts, legal case law, financial reports, scientific code. Measure perplexity on a held-out domain set and downstream task gains on a single domain benchmark.

**Difficulty:** moderate. **Compute:** one consumer GPU sufficient for a 4-hour continued pretraining run.

### Track B — Low-resource language
Train a tokenizer and a small LM from scratch for an under-resourced language (Kurdish, Azerbaijani, Uzbek, Quechua, ...). Evaluate on a single downstream task — sentiment classification, named-entity recognition, or simple machine translation.

**Difficulty:** higher (data collection is the bottleneck). **Compute:** small.

### Track C — Capability evaluation suite
Build a rigorous evaluation suite for one capability of your choice — mathematical reasoning, code generation, factuality, or instruction following. Benchmark three open models on it. Report confidence intervals. Identify contamination where possible.

**Difficulty:** lower technical floor, higher methodology bar. **Compute:** API or inference-only.

You are encouraged to bring your own dataset or downstream task — many of the most interesting projects come from outside the standard benchmarks.

## 2. Deliverables

### A reproducible repository

```
project/
├── data/
│   ├── README.md         # source, licence, preprocessing steps
│   └── prepare.py        # downloads & preprocesses raw data
├── configs/
│   └── base.yaml         # all hyperparameters in version control
├── src/
│   ├── train.py
│   ├── eval.py
│   └── model.py
├── notebooks/
│   ├── 01_eda.ipynb
│   ├── 02_baseline.ipynb
│   └── 03_results.ipynb
├── report.pdf
└── README.md             # one-page summary + how to reproduce
```

Anything not in the repo doesn't exist.

### A 6–10 page report (NeurIPS format)

Required sections: Abstract, Introduction, Method, Experiments, Results, Failure analysis, Limitations, Related work, References. Aim for the tone of an arXiv preprint, not a course assignment.

### A 15-minute presentation

10 minutes of content, 5 minutes Q&A. One slide per minute is the upper bound. The audience is your peers, not a textbook reader — assume they know weeks 1–11.

## 3. The rubric

Grading uses the rubric in `../assets/rubric.md`. Summary:

- **Technical soundness (40 pts)** — problem framing, method selection, implementation quality, evaluation rigor.
- **Scientific communication (30 pts)** — writing, failure analysis, visualizations.
- **Originality and insight (20 pts)** — what did *you* learn that wasn't in the readings?
- **Presentation (10 pts)** — clarity, audience awareness, Q&A handling.

The most common reason for a low score is **a single number with no baseline and no confidence interval.** A small experiment with a proper comparison beats a large experiment without one. Every result table must include at least one sensible baseline.

## 4. A worked planning template

Use this skeleton when drafting your proposal.

In [ ]:
PROPOSAL_TEMPLATE = '''
# Project proposal — <your title>

## Research question
One sentence. A specific claim or measurement, not a topic area.
Bad : "I will study tokenizers."
Good: "Does byte-level BPE recover Turkish morphological boundaries
       better than WordPiece, measured on a 1000-word annotated set?"

## Hypothesis
What outcome do you expect, and why? Be willing to be wrong.

## Method
- Data: source, size, licence, preprocessing.
- Model: architecture and parameter budget. Existing checkpoint, or from scratch?
- Training: optimizer, schedule, total tokens, hardware estimate.
- Evaluation: which benchmark(s)? Which baselines? Confidence intervals how?

## Risks
List the top three things that could go wrong, with mitigations.

## Timeline
Day 1–2: data + EDA.   Day 3–4: baseline + first run.
Day 5–6: ablations + writing.   Day 7: presentation.
'''
print(PROPOSAL_TEMPLATE)

## 5. Anti-patterns to avoid

A short field guide based on common mistakes from previous cohorts.

- **No baseline.** "My model achieves 73% accuracy." Without a comparison number, this is uninterpretable. Always include at least one trivial baseline (majority class, BM25, untrained random init, untuned reference model).
- **No confidence interval.** With small evaluation sets, single-number differences are usually noise. Bootstrap (Week 11) or report per-seed variance.
- **Leakage.** A test split that overlaps the training split, or that was used during hyperparameter selection, invalidates everything. Define train/dev/test splits *before* any modeling.
- **Cherry-picked qualitative examples.** Three good outputs prove nothing. If you show generations, sample uniformly at random.
- **No ablation.** "I tried X and it worked." Which component drove the result? Run the natural lesion experiment.
- **No failure analysis.** A report without failure cases is incomplete. The most interesting page of a paper is usually the error analysis.

## 6. Tools and references

A non-exhaustive list of practical resources you may reach for.

**Data.** Hugging Face Datasets, Common Crawl (filtered), language-specific corpora (e.g. TS Corpus for Turkish), domain-specific dumps (PubMed, S2ORC, The Stack).

**Training.** PyTorch, Hugging Face Transformers + Accelerate, TRL (for RLHF/DPO), DeepSpeed or FSDP for multi-GPU. For tokenizer training, `tokenizers` is the production-grade library; your from-scratch BPE from Week 2 is a teaching aid, not a recommendation for the capstone.

**Evaluation.** lm-evaluation-harness, HELM, BIG-bench. For domain-specific tasks, you may need to build the harness yourself — that itself is a perfectly good capstone (Track C).

**Logging.** Weights & Biases, MLflow, or a plain CSV — anything that lets you reconstruct what you ran a week from now. Reproducibility is a grading criterion.

In [ ]:
# Sanity checklist before submission — fill in honestly.
checklist = {
    "All hyperparameters in a config file (not hard-coded)": False,
    "Random seeds fixed and documented": False,
    "Train/dev/test split defined before modeling": False,
    "At least one non-trivial baseline reported": False,
    "Confidence intervals on every claim": False,
    "Failure cases inspected and documented": False,
    "README explains how to reproduce in one command": False,
    "All figures have axis labels, units, and a caption": False,
    "Cited every external dataset, model, and library": False,
    "Report runs through a spell-checker and a grammar-checker": False,
}
for item, done in checklist.items():
    box = "[x]" if done else "[ ]"
    print(f"  {box} {item}")

## 7. Final note

The point of this capstone is not to ship a state-of-the-art model — twelve weeks isn't enough for that, and frontier labs spend orders of magnitude more compute. The point is to demonstrate that you can plan a focused experiment, execute it carefully, report it honestly, and defend it.

A modest, well-executed experiment beats an ambitious one that overran and reported a single uncalibrated number. Choose scope accordingly. Good luck.